In [2]:
"""
ParkCast SF — Data Ingestion Script
Pulls data from:
  1. SFpark Sensor API        (parking occupancy)
  2. SF Open Data             (street cleaning, meters)
  3. SF Events API            (demand spikes)
  4. Open-Meteo Weather API   (rain/fog)
  5. US Federal Holidays      (holiday patterns)
  6. SF School Calendar       (school day patterns)

Saves merged dataset to:
  - Local CSV: data/parkcast_raw.csv
  - MLflow artifact (logged to GCP MLflow server)
"""
import os

# ── Fix artifact URI before importing mlflow ──────────────────
os.environ["MLFLOW_ARTIFACT_URI"] = "http://34.133.160.231:5000"

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import mlflow
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://34.133.160.231:5000"   # replace with your GCP IP
EXPERIMENT_NAME     = "parkcast-data-ingestion"
DATA_DIR            = "data"
OUTPUT_FILE         = os.path.join(DATA_DIR, "parkcast_raw.csv")

# Date range to pull (last 90 days for a solid training set)
END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=90)
DATE_FMT   = "%Y-%m-%d"

os.makedirs(DATA_DIR, exist_ok=True)

# ── MLflow setup ──────────────────────────────────────────────────────────────
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 1: SFpark Sensor API
# Docs: https://www.sfmta.com/getting-around/drive-park/demand-responsive-pricing/sfpark-pilot-overview
# ─────────────────────────────────────────────────────────────────────────────
def fetch_sfpark_data():
    """
    Fetch parking occupancy data from SFpark API.
    Returns a DataFrame with columns:
      block_id, street, neighborhood, lat, lon,
      timestamp, occupied_spaces, total_spaces, occupancy_pct
    """
    print("Fetching SFpark sensor data...")

    url = "http://api.sfpark.org/sfpark/rest/availabilityservice"
    params = {
        "response": "json",
        "uom": "mile",
        "motorist_type": "1",
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        records = []
        for loc in data.get("AVL", {}).get("loc", []):
            try:
                records.append({
                    "block_id":        loc.get("NAME", "unknown"),
                    "street":          loc.get("DESC", "unknown"),
                    "neighborhood":    loc.get("OPS", "unknown"),
                    "lat":             float(loc.get("LAT", 0)),
                    "lon":             float(loc.get("LNG", 0)),
                    "timestamp":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "occupied_spaces": int(loc.get("OPON", 0)),
                    "total_spaces":    int(loc.get("OPHRS", 1)),
                    "occupancy_pct":   float(loc.get("OPON", 0)) / max(float(loc.get("OPHRS", 1)), 1) * 100,
                })
            except (ValueError, TypeError):
                continue

        df = pd.DataFrame(records)
        print(f"  SFpark: {len(df)} records fetched")
        return df

    except Exception as e:
        print(f"  SFpark API unavailable ({e}), generating synthetic data for development...")
        return _generate_synthetic_sfpark_data()


def _generate_synthetic_sfpark_data():
    """
    Generate realistic synthetic SFpark data for development/testing
    when the API is unavailable. Mirrors real SFpark structure.
    """
    np.random.seed(42)

    neighborhoods = ["Mission", "SoMa", "Castro", "Marina", "Tenderloin",
                     "Haight", "Noe Valley", "Sunset", "Richmond", "SOMA"]
    streets = ["Valencia St", "Market St", "Castro St", "Chestnut St",
               "Divisadero St", "Haight St", "24th St", "Irving St",
               "Clement St", "Folsom St"]

    records = []
    # Generate hourly data for the past 90 days
    current = START_DATE
    while current <= END_DATE:
        for hour in range(24):
            for i, neighborhood in enumerate(neighborhoods):
                # Simulate realistic occupancy patterns
                # Peak hours: 9-11am and 6-8pm on weekdays
                is_weekday = current.weekday() < 5
                base_occ = 0.4

                if is_weekday:
                    if 9 <= hour <= 11:
                        base_occ = 0.85
                    elif 18 <= hour <= 20:
                        base_occ = 0.90
                    elif 12 <= hour <= 14:
                        base_occ = 0.70
                    elif 0 <= hour <= 6:
                        base_occ = 0.15
                else:
                    if 11 <= hour <= 15:
                        base_occ = 0.75
                    elif 19 <= hour <= 22:
                        base_occ = 0.85

                # Add some noise
                occ = min(1.0, max(0.0, base_occ + np.random.normal(0, 0.1)))
                total = np.random.randint(20, 80)
                occupied = int(occ * total)

                records.append({
                    "block_id":        f"block_{i+1:03d}",
                    "street":          streets[i % len(streets)],
                    "neighborhood":    neighborhood,
                    "lat":             37.7749 + np.random.uniform(-0.05, 0.05),
                    "lon":             -122.4194 + np.random.uniform(-0.05, 0.05),
                    "timestamp":       current.replace(hour=hour).strftime("%Y-%m-%d %H:%M:%S"),
                    "occupied_spaces": occupied,
                    "total_spaces":    total,
                    "occupancy_pct":   round(occ * 100, 2),
                })
        current += timedelta(days=1)

    df = pd.DataFrame(records)
    print(f"  Synthetic SFpark: {len(df)} records generated")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 2: SF Open Data — Street Cleaning Schedule
# Docs: https://data.sfgov.org/Transportation/Street-Sweeping-Schedule/yhqp-tvqs
# ─────────────────────────────────────────────────────────────────────────────
def fetch_street_cleaning():
    """
    Fetch street cleaning schedule from SF Open Data.
    Returns a DataFrame indicating which blocks have cleaning on which days/hours.
    """
    print("Fetching street cleaning schedule...")

    url = "https://data.sfgov.org/resource/yhqp-tvqs.json"
    params = {"$limit": 5000}

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        records = []
        for item in data:
            records.append({
                "street":        item.get("streetname", "unknown"),
                "cleaning_day":  item.get("weekday", "unknown"),
                "start_hour":    item.get("fromhour", 0),
                "end_hour":      item.get("tohour", 0),
                "side":          item.get("cnnrightleft", "unknown"),
            })

        df = pd.DataFrame(records)
        print(f"  Street cleaning: {len(df)} records fetched")
        return df

    except Exception as e:
        print(f"  Street cleaning API unavailable ({e}), using empty DataFrame")
        return pd.DataFrame(columns=["street", "cleaning_day", "start_hour", "end_hour", "side"])


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 3: SF Events API
# Docs: https://data.sfgov.org/Culture-and-Recreation/Film-Locations-in-San-Francisco/yitu-d5am
# ─────────────────────────────────────────────────────────────────────────────
def fetch_sf_events():
    """
    Fetch public events from SF Open Data.
    Returns a DataFrame with event dates that drive parking demand spikes.
    """
    print("Fetching SF events data...")

    url = "https://data.sfgov.org/resource/pyih-qa8i.json"
    params = {"$limit": 1000}

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        records = []
        for item in data:
            records.append({
                "event_name":  item.get("permit_type", "unknown"),
                "event_date":  item.get("start_date_time", "")[:10],
                "neighborhood": item.get("neighborhoods_sic_district", "unknown"),
                "has_event":   1,
            })

        df = pd.DataFrame(records)
        df = df[df["event_date"] != ""]
        print(f"  SF Events: {len(df)} records fetched")
        return df

    except Exception as e:
        print(f"  SF Events API unavailable ({e}), generating synthetic events...")
        return _generate_synthetic_events()


def _generate_synthetic_events():
    """Generate synthetic event data for development."""
    records = []
    # Simulate Giants home games and major events
    current = START_DATE
    while current <= END_DATE:
        # ~30% chance of an event on any given day
        if np.random.random() < 0.3:
            records.append({
                "event_name":   np.random.choice(["Giants Game", "Concert", "Festival", "Parade"]),
                "event_date":   current.strftime(DATE_FMT),
                "neighborhood": np.random.choice(["SoMa", "Mission", "Castro", "Marina"]),
                "has_event":    1,
            })
        current += timedelta(days=1)

    df = pd.DataFrame(records)
    print(f"  Synthetic Events: {len(df)} records generated")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 4: Open-Meteo Weather API
# Docs: https://open-meteo.com/en/docs
# ─────────────────────────────────────────────────────────────────────────────
def fetch_weather():
    """
    Fetch hourly weather data for San Francisco from Open-Meteo.
    Returns a DataFrame with rain and temperature by hour.
    """
    print("Fetching weather data from Open-Meteo...")

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":        37.7749,
        "longitude":       -122.4194,
        "start_date":      START_DATE.strftime(DATE_FMT),
        "end_date":        END_DATE.strftime(DATE_FMT),
        "hourly":          "temperature_2m,precipitation,weathercode",
        "timezone":        "America/Los_Angeles",
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        hourly = data.get("hourly", {})
        df = pd.DataFrame({
            "timestamp":   hourly.get("time", []),
            "temperature": hourly.get("temperature_2m", []),
            "precipitation": hourly.get("precipitation", []),
            "weather_code": hourly.get("weathercode", []),
        })

        # Create binary rain flag (precipitation > 0.1mm)
        df["is_raining"] = (df["precipitation"] > 0.1).astype(int)
        # Weather code >= 51 = drizzle/rain/snow
        df["bad_weather"] = (df["weather_code"] >= 51).astype(int)

        print(f"  Weather: {len(df)} hourly records fetched")
        return df

    except Exception as e:
        print(f"  Weather API unavailable ({e}), generating synthetic weather...")
        return _generate_synthetic_weather()


def _generate_synthetic_weather():
    """Generate synthetic SF weather data (SF is mostly mild/foggy)."""
    records = []
    current = START_DATE
    while current <= END_DATE:
        for hour in range(24):
            # SF: rain mostly in winter months (Nov-Mar)
            rain_prob = 0.3 if current.month in [11, 12, 1, 2, 3] else 0.05
            is_raining = 1 if np.random.random() < rain_prob else 0
            records.append({
                "timestamp":     current.replace(hour=hour).strftime("%Y-%m-%d %H:%M:%S"),
                "temperature":   np.random.uniform(50, 70),
                "precipitation": np.random.uniform(0, 5) if is_raining else 0,
                "weather_code":  61 if is_raining else 0,
                "is_raining":    is_raining,
                "bad_weather":   is_raining,
            })
        current += timedelta(days=1)

    df = pd.DataFrame(records)
    print(f"  Synthetic Weather: {len(df)} records generated")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 5: US Federal Holidays
# ─────────────────────────────────────────────────────────────────────────────
def fetch_holidays():
    """
    Fetch US federal holidays using a public API.
    Returns a DataFrame with holiday dates.
    """
    print("Fetching US federal holidays...")

    url = "https://date.nager.at/api/v3/PublicHolidays/2025/US"

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()

        df = pd.DataFrame([{
            "date":         item["date"],
            "holiday_name": item["name"],
            "is_holiday":   1,
        } for item in data])

        print(f"  Holidays: {len(df)} records fetched")
        return df

    except Exception as e:
        print(f"  Holidays API unavailable ({e}), using hardcoded 2025 holidays...")
        holidays = [
            "2025-01-01", "2025-01-20", "2025-02-17", "2025-05-26",
            "2025-06-19", "2025-07-04", "2025-09-01", "2025-10-13",
            "2025-11-11", "2025-11-27", "2025-12-25",
        ]
        return pd.DataFrame({"date": holidays, "holiday_name": "Holiday", "is_holiday": 1})


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 6: SF School Calendar (SFUSD)
# ─────────────────────────────────────────────────────────────────────────────
def fetch_school_calendar():
    """
    Generate SF school calendar (SFUSD school days affect neighborhood parking).
    Returns a DataFrame with school day indicators.
    """
    print("Generating SF school calendar...")

    # SFUSD typically runs Aug-Jun, Mon-Fri
    records = []
    current = START_DATE
    while current <= END_DATE:
        is_school_month = current.month in [8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6]
        is_weekday = current.weekday() < 5
        is_school_day = 1 if (is_school_month and is_weekday) else 0

        records.append({
            "date":          current.strftime(DATE_FMT),
            "is_school_day": is_school_day,
        })
        current += timedelta(days=1)

    df = pd.DataFrame(records)
    print(f"  School calendar: {len(df)} days generated")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# MERGE: Combine all sources into one feature-rich dataset
# ─────────────────────────────────────────────────────────────────────────────
def merge_all_sources(sfpark_df, street_cleaning_df, events_df,
                       weather_df, holidays_df, school_df):
    """
    Merge all data sources into a single training-ready DataFrame.
    """
    print("\nMerging all data sources...")

    df = sfpark_df.copy()

    # Parse timestamp
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["date"]      = df["timestamp"].dt.strftime(DATE_FMT)
    df["hour"]      = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek   # 0=Monday
    df["month"]     = df["timestamp"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_rush_hour"] = df["hour"].apply(
        lambda h: 1 if (7 <= h <= 9 or 17 <= h <= 19) else 0
    )

    # Merge weather
    if not weather_df.empty:
        weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
        weather_df["date"] = weather_df["timestamp"].dt.strftime(DATE_FMT)
        weather_df["hour"] = weather_df["timestamp"].dt.hour
        df = df.merge(
            weather_df[["date", "hour", "temperature", "is_raining", "bad_weather"]],
            on=["date", "hour"], how="left"
        )
    else:
        df["is_raining"] = 0
        df["bad_weather"] = 0
        df["temperature"] = 60.0

    # Merge events (by date)
    if not events_df.empty:
        events_agg = events_df.groupby("event_date")["has_event"].max().reset_index()
        events_agg.columns = ["date", "has_nearby_event"]
        df = df.merge(events_agg, on="date", how="left")
        df["has_nearby_event"] = df["has_nearby_event"].fillna(0).astype(int)
    else:
        df["has_nearby_event"] = 0

    # Merge holidays
    if not holidays_df.empty:
        df = df.merge(
            holidays_df[["date", "is_holiday"]],
            on="date", how="left"
        )
        df["is_holiday"] = df["is_holiday"].fillna(0).astype(int)
    else:
        df["is_holiday"] = 0

    # Merge school calendar
    if not school_df.empty:
        df = df.merge(school_df, on="date", how="left")
        df["is_school_day"] = df["is_school_day"].fillna(0).astype(int)
    else:
        df["is_school_day"] = 0

    # Street cleaning flag (simplified: mark known cleaning hours)
    df["is_street_cleaning"] = df["hour"].apply(
        lambda h: 1 if 8 <= h <= 12 else 0
    )

    # Drop rows with missing target
    df = df.dropna(subset=["occupancy_pct"])

    # Final feature set
    feature_cols = [
        "block_id", "street", "neighborhood", "lat", "lon",
        "date", "hour", "day_of_week", "month",
        "is_weekend", "is_rush_hour", "is_street_cleaning",
        "has_nearby_event", "is_holiday", "is_school_day",
        "is_raining", "bad_weather", "temperature",
        "total_spaces", "occupancy_pct"   # occupancy_pct = target
    ]

    # Keep only columns that exist
    feature_cols = [c for c in feature_cols if c in df.columns]
    df = df[feature_cols]

    print(f"  Merged dataset: {len(df)} rows, {len(df.columns)} columns")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("ParkCast SF — Data Ingestion")
    print(f"Date range: {START_DATE.strftime(DATE_FMT)} to {END_DATE.strftime(DATE_FMT)}")
    print("=" * 60)

    with mlflow.start_run(run_name="data-ingestion"):

        # ── Fetch all sources ─────────────────────────────────────────
        sfpark_df         = fetch_sfpark_data()
        street_cleaning_df = fetch_street_cleaning()
        events_df         = fetch_sf_events()
        weather_df        = fetch_weather()
        holidays_df       = fetch_holidays()
        school_df         = fetch_school_calendar()

        # ── Merge ─────────────────────────────────────────────────────
        final_df = merge_all_sources(
            sfpark_df, street_cleaning_df, events_df,
            weather_df, holidays_df, school_df
        )

        # ── Save locally ──────────────────────────────────────────────
        final_df.to_csv(OUTPUT_FILE, index=False)
        print(f"\nSaved to {OUTPUT_FILE}")

        # ── Log to MLflow ─────────────────────────────────────────────
        mlflow.log_params({
            "start_date":     START_DATE.strftime(DATE_FMT),
            "end_date":       END_DATE.strftime(DATE_FMT),
            "num_rows":       len(final_df),
            "num_features":   len(final_df.columns),
            "data_sources":   "sfpark,street_cleaning,events,weather,holidays,school",
        })

        mlflow.log_metrics({
            "total_rows":         len(final_df),
            "missing_pct":        round(final_df.isnull().mean().mean() * 100, 2),
            "avg_occupancy_pct":  round(final_df["occupancy_pct"].mean(), 2),
            "num_neighborhoods":  final_df["neighborhood"].nunique(),
            "num_blocks":         final_df["block_id"].nunique(),
        })

        #mlflow.log_artifact(OUTPUT_FILE)

        # ── Summary ───────────────────────────────────────────────────
        print("\n" + "=" * 60)
        print("INGESTION COMPLETE")
        print("=" * 60)
        print(f"Total rows:          {len(final_df):,}")
        print(f"Total features:      {len(final_df.columns)}")
        print(f"Neighborhoods:       {final_df['neighborhood'].nunique()}")
        print(f"Unique blocks:       {final_df['block_id'].nunique()}")
        print(f"Avg occupancy:       {final_df['occupancy_pct'].mean():.1f}%")
        print(f"Missing data:        {final_df.isnull().mean().mean()*100:.2f}%")
        print(f"\nFeatures: {list(final_df.columns)}")
        print(f"\nSample data:")
        print(final_df.head(3).to_string())
        print("=" * 60)


if __name__ == "__main__":
    main()

ParkCast SF — Data Ingestion
Date range: 2026-01-02 to 2026-04-02
Fetching SFpark sensor data...
  SFpark API unavailable (HTTPConnectionPool(host='api.sfpark.org', port=80): Max retries exceeded with url: /sfpark/rest/availabilityservice?response=json&uom=mile&motorist_type=1 (Caused by NameResolutionError("HTTPConnection(host='api.sfpark.org', port=80): Failed to resolve 'api.sfpark.org' ([Errno 8] nodename nor servname provided, or not known)"))), generating synthetic data for development...
  Synthetic SFpark: 21840 records generated
Fetching street cleaning schedule...
  Street cleaning API unavailable (404 Client Error: Not Found for url: https://data.sfgov.org/resource/yhqp-tvqs.json?%24limit=5000), using empty DataFrame
Fetching SF events data...
  SF Events: 0 records fetched
Fetching weather data from Open-Meteo...
  Weather: 2184 hourly records fetched
Fetching US federal holidays...
  Holidays: 16 records fetched
Generating SF school calendar...
  School calendar: 91 days g